**Module**: Data 9910 – Working with Data

**Lecturer**: Lucas Rizzo

**Class Group**: TU059/060/256

**Worth**: 20%

**Due Time**: Before next class on Week 6.

**Programming Language**: The test can be done in both `R` or `Python`. You can change the kernel language going in *Kernel > Change Kernel*


### Submission

Please submit this notebook and additional files (if any) in a zip file. ***Submissions without a report won't be accepted.***

# Problem description

A telecommunications company is starting a data project to improve their customer analytics capabilities. During informal interviews with the stakeholders, the following were suggested as possible features for the system:

- identify trends by certain types of customers
- identify trends by certain types of calls
- identify relevant statistical information of overall calls performed by customers


# Dataset

The company has provided you with an sql dump in two files (`db.ddl` and `data.sql`) to replicate their PostgreSQL database. However, they are not very organised and did not provide a documentation alongside it. Hence, you will be required to recreate it and do some investigation to understand their data. Note that a ddl file also containts sql commands, but only those that describes the portion of SQL that creates, alters, and deletes database objects

### Create a new local database

Using pgAdmin create a new server to run the sql files.

![Screenshot of pgAdmin 4 to create a new connection](new_connection.jpg)

You can use the parameters below to create the databse. If you use different ones you will need to remember them to connect to the database.

`user = 'postgres'`

`password = 'admin'`

`host = 'localhost'`

`port = 5432`

`database = 'telecommunications'`


### Import the data

Using the query tool open the `db.ddl` file and run the commands to create the necessary tables. After that, open the `data.sql` to run the insert commands and add all the data to the database (this can take around 20 seconds to run).

![Screenshot of pgAdmin 4 to open a SQL file in the query tool](query_tool.jpg)

### QUESTION 1

Using your preferred language (R or Python) create a connection object to the companies database. (1 mark)

In [35]:
# import required packages
import sqlalchemy as db
from sqlalchemy import text
import pandas as pd
import psycopg2

In [76]:
# create a connection to the telecommunications database
user = 'postgres'
password = 'password0134!'
host = 'localhost'
port = 5432
database = 'telecommunications'

engine = db.create_engine('postgresql+psycopg2://{0}:{1}@{2}:{3}/{4}'.format(user,
                                                                    password,
                                                                    host,
                                                                    port,
                                                                    database))
conn = engine.connect()


### QUESTION 2

Extract the ERD from the database (this can be done via PgAdmin or via sqlalchemy_schemadisplay). Write a general description of the database structure (2 marks)


Below is an overview of the **telecommunications** database and  its Entity Relationship Diagram (ERD).

### Tables
There are 4 tables present in the database
- *call_rates*
    - Primary Key = *callrate* - this is a unique identifier
    - Foreign Key = None - no constraints on tables values
    - Attributes and datatype =
        - *callrate* = int
        - *isinternatioal* = VARCHAR(5)
        - *isroaming* = VARCHAR(5)
        - *costperminute* = numeric(4,2) precision and scale values (00.00) 
- *calls*
    - Primary Key = *connectionid* - this is a unique identifier
    - Foreign Keys = *call_rates.callrate* + *customers.phonenumber*
    - attributes =
        - *phonenumber* = VARCHAR(13)
        - *calltime* = VARCHAR(16)
        - *duration* = numeric(13,9) precision and scale values (0000.000000000) 
        - *connectionid* = VARCHAR(36)
        - *callrate* = int
- *customers*
    - Primary Key = *phonenumber* - this is a unique identifier
    - Foreign Key = None - no constraints on tables values
    - attributes =
        - *phonenumber* = VARCHAR(13)
        - *contractstartdate* = date
        - *dob* = date
- *customer_service*
    - Primary Key = *connectionid* - this is a unique identifier
    - Foreign Key = calls.connectionid - 
    - attributes =
        - *connectionid* = VARCHAR(36)

### Relationships
- The *calls* table is a bridging entity between the customers and call_rates tables.
    - *callrate* could have many *calls* associated with it but a *call* can have only one *callrate*
    - *customers* could have many *calls* but a *call* can only have one *customer*
- The *customer_service* table has a one to one relationship with the *calls* table
    - a *customer_service* entry can only have one *call* and *call* can only have one *customer_service* id

### shape and structure
- Max number of rows  = 1000 rows in *customer_service*,  *customers* and *calls*.
- max number of columns = 5 in calls.


![telecommunication database erd diagram](./tcomm-erd-2.png)
ERD diagram


In [59]:
# shape and structure
# getign errors on connectin so closing and openign the connection hagain before runnign to ensure connection is open and available
# ideally woudl close conenctin after each query
conn.rollback()

# Define your tables
db_table_names = ["customers", "calls", "customer_service", "call_rates"]

# Loop through each table, run a count query, and print results
for table in db_table_names:
    result = conn.execute(text(f"SELECT COUNT(*) FROM {table};"))
    count = result.fetchone()[0]  # get the first (and only) value
    print(f"Table '{table}' has {count} rows.")




Table 'customers' has 4999 rows.
Table 'calls' has 161584 rows.
Table 'customer_service' has 16189 rows.
Table 'call_rates' has 4 rows.


In [68]:
conn.rollback()

db_table_names = ["customers", "calls", "customer_service", "call_rates"]

# created dictionary to store dataframe for the tables
table_dict={}

# from content and also from here
# > https://pynative.com/python-cursor-fetchall-fetchmany-fetchone-to-read-rows-from-table/
for table in db_table_names:
        result = conn.execute(text("select * from {table}")).fetchall()
        tabledf = pd.DataFrame(result)
        tabledict = tabledf
        

# data.shape
# data.head()

# Load data
# Print shape for each table
for table, df in table_data.items():
    print(f"Table '{table}' has {df.shape[0]} rows and {df.shape[1]} columns.")


ProgrammingError: (psycopg2.errors.SyntaxError) syntax error at or near "{"
LINE 1: select * from {table}
                      ^

[SQL: select * from {table}]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [82]:
student = db.Table('call_rates', metadata_obj, autoload=True, autoload_with=engine)
stmt = db.select(student) # select statement in SQLALchemy
print(stmt)
print()
# execute the statement on connection and fetch 10 records: results
results = conn.execute(stmt).fetchall()
print(results)


SELECT call_rates.callrate, call_rates.isinternational, call_rates.isroaming, call_rates.costperminute 
FROM call_rates

[(1, 'TRUE', 'TRUE', Decimal('0.57')), (2, 'TRUE', 'FALSE', Decimal('0.26')), (3, 'FALSE', 'TRUE', Decimal('0.26')), (4, 'FALSE', 'FALSE', Decimal('0.03'))]


In [43]:
df_customers = pd.read_sql(text("SELECT * FROM customers;"), conn)

# Replace '?' and 'NA' with real NaN
df_customers = df_customers.replace(['?', 'NA', 'N/A'], pd.NA)

# Count missing per column
missing_summary = df_customers.isna().sum()

print(missing_summary)


phonenumber          0
contractstartdate    0
dob                  0
dtype: int64


In [49]:
tables = ["customers", "calls", "customer_service", "call_rates"]

summary = []

for table in tables:
    print(f"\n🔍 Checking table: {table}")
    df = pd.read_sql(text(f"SELECT * FROM {table};"), conn)

    # Count NULL, 'NA', and '?'
    nulls = df.isnull().sum().sum()
    nas = (df == 'NA').sum(numeric_only=False).sum()
    qs = (df == '?').sum(numeric_only=False).sum()
    invalid_total = nulls + nas + qs
    
    
    
    # Count completely duplicate rows
    duplicate_rows = df.duplicated().sum()

    summary.append({
        "table_name": table,
        "invalid_values": invalid_total,
        "duplicate_rows": duplicate_rows,
        "total_rows": len(df)
    })

df_summary = pd.DataFrame(summary)
print(df_summary)

print ("Rows     : " ,df.shape[0])
print ("Columns  : " ,df.shape[1])
print('------')
df.describe().transpose()


🔍 Checking table: customers

🔍 Checking table: calls

🔍 Checking table: customer_service

🔍 Checking table: call_rates
         table_name  invalid_values  duplicate_rows  total_rows
0         customers               0               0        4999
1             calls               0               0      161584
2  customer_service               0               0       16189
3        call_rates               0               0           4
Rows     :  4
Columns  :  4
------


,count,mean,std,min,25%,50%,75%,max
callrate,4.0,2.50,1.290994,1.00,1.7500,2.50,3.2500,4.00
costperminute,4.0,0.28,0.221660,0.03,0.2025,0.26,0.3375,0.57


# Customer information

Query the database to extract the following information:

a) The overall number of customers (1 mark)

In [ ]:
# initialising the metadata aobject 
metadata_obj = db.MetaData() # initialize
metadata_obj.reflect(bind=engine) # reflect
for t in metadata_obj.sorted_tables:
    print(t)

In [ ]:
result = conn.execute(text("SELECT * FROM customers"))

# print all results
for row in result:
    print(row)


InternalError: (psycopg2.errors.InFailedSqlTransaction) current transaction is aborted, commands ignored until end of transaction block

[SQL: SELECT * FROM customers]
(Background on this error at: https://sqlalche.me/e/20/2j85)

In [26]:
# result = conn.execute(text("SELECT COUNT(*) FROM customers"))

# print(result)
# # # print all results
# for row in result:
#     print(row)
# # Fetch the single result row
# total_customers = result.scalar()

# print("Total number of customers:", total_customers)

# inspector = inspect(engine)
# tables = inspector.get_table_names()

# print("Number of tables:", len(tables))
# print("Table names:", tables)

# Run the query and load directly into a pandas DataFrame
# df_total = pd.read_sql(text("SELECT COUNT(*) AS total_customers FROM customers;"), conn)

# # Display the DataFrame nicely
# print(df_total)


#pd.read_sql(text("SELECT table_schema, table_name FROM information_schema.tables WHERE table_name = 'customers';"), conn)
df_check = pd.read_sql(text("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT phonenumber) AS unique_customers
FROM customers;
"""), conn)

print(df_check)




   total_rows  unique_customers
0        4999              4999


In [ ]:
# a customer could have more than one phone
# but we will just determin the number of customers regardless of the number of unique phone numbers
# do they match?
Select count
cust_count_df <- dbGetQuery(con, "SELECT COUNT(student_number) FROM student WHERE dob > '01-Nov-1980'")
head(s_count_df)




b) The number of customers by year of DOB in descending order (2 marks)

c) Analyse the query below and describe what is the result given by it. (1 mark)

In [25]:
# Python (uncomment if you are using Python)
result = conn.execute(text("SELECT c.phonenumber, SUM(cr.costperminute * c.duration) AS total_amount_spent " + 
                      "FROM public.calls c INNER JOIN public.call_rates cr ON c.callrate = cr.callrate "
                      "WHERE c.connectionid NOT IN ( SELECT DISTINCT connectionid FROM public.customer_service) " +
                      "GROUP BY c.phonenumber ORDER BY c.phonenumber;"))
for row in result:
    print(row)

# R (uncomment if you are using R)
# query <- "SELECT c.phonenumber, SUM(cr.costperminute * c.duration) AS total_amount_spent"
# query <- paste(query, "FROM public.calls c INNER JOIN public.call_rates cr ON c.callrate = cr.callrate")
# query <- paste(query, "WHERE c.connectionid NOT IN ( SELECT DISTINCT connectionid FROM public.customer_service)")
# query <- paste(query, "GROUP BY c.phonenumber ORDER BY c.phonenumber;")

# result <- dbGetQuery(con, query)

# result

('01 000 9436', Decimal('3201.22279225500'))
('01 001 6896', Decimal('11069.16794494900'))
('01 001 9955', Decimal('3143.07759377700'))
('01 002 1488', Decimal('7667.06401183800'))
('01 002 1807', Decimal('1037.42097668100'))
('01 002 3138', Decimal('10969.08258640800'))
('01 002 7003', Decimal('25476.58783680990'))
('01 003 5862', Decimal('2178.43534643600'))
('01 004 1518', Decimal('1583.04664798870'))
('01 004 3369', Decimal('1893.67951344500'))
('01 004 6123', Decimal('3919.39081954100'))
('01 005 3059', Decimal('6618.55011914400'))
('01 005 7353', Decimal('2169.99057156000'))
('01 005 8909', Decimal('16340.42254020600'))
('01 005 9012', Decimal('13579.88341802100'))
('01 006 5619', Decimal('9908.71578357400'))
('01 006 8032', Decimal('6544.87565328800'))
('01 007 0548', Decimal('10625.15229737000'))
('01 007 3126', Decimal('16346.97708737820'))
('01 008 0438', Decimal('11732.57507735700'))
('01 008 1378', Decimal('16202.03157874900'))
('01 008 2702', Decimal('2093.73442266500'))
(

d) There is a logical error on the result of the above query. As a telecommunications specialist, you have observed that the `total_amount_spent` returned is too high for the usual expected values in this industry. Please fix the query so that values falls in a reasonable range. (1 mark)

e) Given the correct query for the `total_amount_spent` spent, identify the min, max and average revenue gererate by overall phone numbers (1 mark)

# Calls information

### Question 4

Query the database to extract the following information:

a) The average call duration in minutes per phone number, excluding those made to the customer service (2 marks)

b) The number of calls made to customer support by phone number (2 marks)

c) The number of calls made by call rate (2 marks)

# Report

Describe your findings in a short report, summarizing the information you have collected from previous questions and giving some interpretation of results. You might for example generate a table with results, and give some interpretation in a few sentences. You can also imagine you are an analyst interested in reading this report and collecting the most important information extracted from the data. (5 marks)

In [44]:
# Add any code here

Add any text here

we dont' have a customer ID number! only way to know how many individual customers there are is by dob
how many dob are tjhe sidentical? 

how many customers have more than one phone?

the most profitable rates
the most used rates

international calls

% roaming percentage of calls overall that are internation - canwe  determine profits off them?

age profile of users?

typical contract start date - can we crete alerts before contract expires to contact customers and try get themto rne before expiry
to keep their numbers they need to renww before expiry otherwise we issue them wirha  new numbers (etoprtion)